In [1]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

In [2]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
Using num_workers = 0


In [3]:
def evaluate_model(modelpath, modeltype, verbose=True):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=512, 
        dropout_rate=0.3, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [4]:
def evaluate_multiple_models(modeldict, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype)
        collected_results[modelname] = results
    return collected_results

In [5]:
basepath = os.path.join(os.path.expanduser("~"), "Downloads")

modeldict = {
    "Asymmetric": os.path.join(basepath, "Model_Asymm_Loss", "best_model.pth"),
    "Focal": os.path.join(basepath, "Model_Focal_Loss", "best_model.pth"), 
    "BCE_Weighted": os.path.join(basepath, "Model_BCE_Weighted", "best_model.pth"), 
    "BCE_Unweighted": os.path.join(basepath, "Model_BCE_Unweighted", "best_model.pth")
}

agg_results = evaluate_multiple_models(modeldict)

Loading from checkpoint, last run epoch was 10


Loading from checkpoint, last run epoch was 12


Loading from checkpoint, last run epoch was 14


Loading from checkpoint, last run epoch was 9


In [6]:
from tabulate import tabulate
models = agg_results.keys()
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name        Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
--------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
Asymmetric        0.750504           0.740675        0.76331     0.246277     0.2443         0.136613        0.935022       0.940098    0.817733    0.822121
Focal             0.753916           0.737279        0.773472    0.0959412    0.0877788      0.0783935       0.937321       0.940961    0.820449    0.813308
BCE_Weighted      0.764529           0.752629        0.780715    0.128215     0.128215       0.0913071       0.939297       0.935208    0.820398    0.799353
BCE_Unweighted    0.748537           0.757159        0.7441      0.0270799    0.0132311      0.0675942       0.932502       0.938453    0.815071    0.814574


In [7]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
--------------  ---------  --------  --------  --------  --------
Asymmetric       0.749517  0.627778  0.759413  0.768588  0.847222
Focal            0.745443  0.624658  0.756825  0.763222  0.879433
BCE_Weighted     0.759318  0.645517  0.7519    0.780063  0.885845
BCE_Unweighted   0.753604  0.628912  0.746861  0.765534  0.847775


In [8]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
--------------  ----------------  ---------------  --------------  --------------  --------------
Asymmetric              0.7046           0.645714        0.73913         0.804196        0.809735
Focal                   0.686343         0.633333        0.733087        0.77649         0.857143
BCE_Weighted            0.702103         0.659155        0.794161        0.771518        0.836207
BCE_Unweighted          0.71965          0.700997        0.75233         0.793814        0.819005


In [9]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
--------------  -------------  ------------  -----------  -----------  -----------
Asymmetric           0.80055       0.610811     0.78084        0.736      0.88835
Focal                0.815681      0.616216     0.782152       0.7504     0.902913
BCE_Weighted         0.826685      0.632432     0.713911       0.7888     0.941748
BCE_Unweighted       0.790922      0.57027      0.74147        0.7392     0.878641


In [10]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
--------------  ------------  -----------  ----------  ----------  ----------
Asymmetric          0.934912     0.903579    0.921542    0.928175    0.986902
Focal               0.933508     0.911224    0.919911    0.931868    0.990095
BCE_Weighted        0.9364       0.916804    0.922053    0.935467    0.985763
BCE_Unweighted      0.935578     0.896475    0.922377    0.922578    0.985503
